In [4]:

from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\User\Desktop\EduInsight-AI")
DATA_DIR = PROJECT_DIR / "data"

# Check and fix folders
for folder_name in ["raw", "processed"]:
    folder = DATA_DIR / folder_name

    if folder.is_file():
        old_name = DATA_DIR / f"{folder_name}_old"
        folder.rename(old_name)
        print(f"Existing file renamed: {old_name}")

    folder.mkdir(parents=True, exist_ok=True)

    print(f"{folder_name}: directory created =", folder.is_dir())

print("\nFolder structure fixed successfully!")


raw: directory created = True
Existing file renamed: C:\Users\User\Desktop\EduInsight-AI\data\processed_old
processed: directory created = True

Folder structure fixed successfully!


In [2]:
pip install nltk

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 3.9 MB/s eta 0:00:01
   ------------------------------------ --- 1.6/1.7 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 4.5 MB/s eta 0:00:00

   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ------------- -------------------------- 1/3 [regex]
   ------------- -------------------------- 1/3 [regex]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ----------


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:

# ============================================================
# EduInsight AI
# Phase 5: NLP Text Preprocessing
# ============================================================

from pathlib import Path
import pandas as pd
import re
import html
import nltk
import warnings

warnings.filterwarnings("ignore")

# 1. PROJECT PATHS

PROJECT_DIR = Path(r"C:\Users\User\Desktop\EduInsight-AI")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 2. LOAD CLEANED DATASETS

data_files = list(PROCESSED_DIR.glob("*_cleaned.csv"))

if not data_files:
    raise FileNotFoundError(
        "No cleaned CSV files found. Run Data Cleaning first."
    )

frames = []

for file in data_files:
    df_part = pd.read_csv(file)
    df_part["source_file"] = file.name
    frames.append(df_part)

df = pd.concat(frames, ignore_index=True)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# 3. IDENTIFY TEXT COLUMNS

text_columns = []

for col in df.select_dtypes(include=["object"]).columns:
    if col == "source_file":
        continue

    average_length = df[col].dropna().astype(str).str.len().mean()

    if pd.notna(average_length) and average_length > 20:
        text_columns.append(col)

print("\nDetected text columns:", text_columns)

if not text_columns:
    raise ValueError(
        "No long text columns detected in the dataset."
    )

# 4. TEXT CLEANING FUNCTION

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove email addresses
    text = re.sub(
        r"\b[\w.-]+@[\w.-]+\.\w+\b",
        " ",
        text
    )

    # Keep letters, numbers and spaces
    text = re.sub(r"[^\w\s]", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

# 5. APPLY TEXT CLEANING

for col in text_columns:
    df[col + "_clean"] = df[col].apply(clean_text)

# 6. TOKENIZATION

def tokenize_text(text):
    return text.split()

for col in text_columns:
    clean_col = col + "_clean"
    df[clean_col + "_tokens"] = df[clean_col].apply(tokenize_text)

# 7. TEXT STATISTICS

for col in text_columns:

    clean_col = col + "_clean"

    df[clean_col + "_word_count"] = (
        df[clean_col].apply(lambda x: len(x.split()))
    )

    print(f"\nStatistics for {col}:")
    print(
        "Average word count:",
        round(df[clean_col + "_word_count"].mean(), 2)
    )

# 8. DISPLAY RESULTS

print("\nOriginal and cleaned text:")

for col in text_columns:
    display(
        df[[col, col + "_clean"]].head(5)
    )

# 9. SAVE PREPROCESSED DATA

output_path = PROCESSED_DIR / "preprocessed_education_feedback.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 50)
print("TEXT PREPROCESSING COMPLETED")
print("=" * 50)
print("Saved to:", output_path)


Dataset shape: (54117, 51)
Columns: ['type', 'source', 'id', 'rating', 'num_token', 'sample_criteria', 'review_text', 'rating_bin', 'rating_cat', 'length_bin', 'source_file', 'df_name', 'count', '%', 'df_name.1', 'length_bin.1', 'count.1', '%.1', 'rating_cat.1', 'rating_bin.1', 'unnamed:_0', 'index', 'department', 'teacher_name', 'token_list', 'entry_idx', 'course_code', 'course_id', 'num_ratings', 'useful', 'easy', 'liked', 'num_reviews', 'rating_category', 'student_id', 'date', 'rating_facilities', 'rating_clubs', 'rating_careerservice', 'rating_internet', 'batch_id', 'course_review', 'course_review_(%)', 'teacher_review', 'teacher_review_(%)', 'university_review', 'university_review_(%)', 'total', 'total_(%)', 'sentence', 'overall']

Detected text columns: ['review_text', 'token_list', 'course_id', 'sentence']

Statistics for review_text:
Average word count: 36.22

Statistics for token_list:
Average word count: 26.57

Statistics for course_id:
Average word count: 1.02

Statistics fo

,review_text,review_text_clean
0,Dr. Zhong is a really good chemistry teacher. ...,dr zhong is a really good chemistry teacher sh...
1,I have her for behav. neuro and she is terribl...,i have her for behav neuro and she is terrible...
2,the prof. is real crap. it is a shame that the...,the prof is real crap it is a shame that the u...
3,He is a great guy and is pretty funny in class...,he is a great guy and is pretty funny in class...
4,"Plain and simple, this guy is HORRIBLE! He not...",plain and simple this guy is horrible he not o...


,token_list,token_list_clean
0,NaN,
1,NaN,
2,NaN,
3,NaN,
4,NaN,


,course_id,course_id_clean
0,NaN,
1,NaN,
2,NaN,
3,NaN,
4,NaN,


,sentence,sentence_clean
0,NaN,
1,NaN,
2,NaN,
3,NaN,
4,NaN,



TEXT PREPROCESSING COMPLETED
Saved to: C:\Users\User\Desktop\EduInsight-AI\data\processed\preprocessed_education_feedback.csv


In [6]:

from pathlib import Path
import pandas as pd
import zipfile
import re
import html

# 1. PATHS

PROJECT_DIR = Path(r"C:\Users\User\Desktop\EduInsight-AI")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

for folder in [RAW_DIR, PROCESSED_DIR]:
    if folder.is_file():
        raise ValueError(f"{folder} is a file, not a directory.")
    folder.mkdir(parents=True, exist_ok=True)

# 2. EXTRACT ZIP FILES

for zip_path in RAW_DIR.rglob("*.zip"):
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(RAW_DIR)

# 3. FIND DATA FILES

extensions = [".csv", ".xlsx", ".xls", ".json"]

data_files = [
    f for f in RAW_DIR.rglob("*")
    if f.is_file() and f.suffix.lower() in extensions
]

print("Files found:", len(data_files))

# 4. LOAD AND CLEAN

def clean_text(value):
    if pd.isna(value):
        return value

    value = html.unescape(str(value))
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"http\S+|www\S+", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()

for file in data_files:

    try:
        if file.suffix.lower() == ".csv":
            df = pd.read_csv(file)

        elif file.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(file)

        else:
            df = pd.read_json(file)

        df.columns = [
            str(c).strip().lower().replace(" ", "_")
            for c in df.columns
        ]

        df = df.dropna(how="all")
        df = df.dropna(axis=1, how="all")
        df = df.drop_duplicates()

        for col in df.select_dtypes(include="object").columns:
            df[col] = df[col].apply(clean_text)

        output = PROCESSED_DIR / f"{file.stem}_cleaned.csv"

        df.to_csv(output, index=False, encoding="utf-8-sig")

        print("Saved:", output.name, "| Shape:", df.shape)

    except Exception as error:
        print("Could not process:", file.name, "|", error)

print("\nDATA CLEANING FINISHED!")


Files found: 214
Could not process: ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21-12)_(FULL_checked)_(V2).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=10)_(2024-11-21_23-08-33)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=11)_(2024-11-22_11-27-55)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=12)_(2024-11-22_11-27-55)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=13)_(2024-11-22_12-24-24)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=14)_(2024-11-23_10-41-02)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=15)_(2024-11-23_10-41-02)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=16)_(2024-11-23_12-11-09)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batc

In [1]:

from pathlib import Path
import json

project = Path.cwd()

if not (project / "data" / "raw").exists():
    project = project.parent

raw_folder = project / "data" / "raw"

files = sorted(
    [
        f for f in raw_folder.rglob("*.json")
        if f.name != "dataset_inspection.json"
    ],
    key=lambda f: f.stat().st_size,
    reverse=True
)

print("JSON fayllarının sayı:", len(files))
print("\nƏN BÖYÜK 5 JSON FAYLI:")

for file in files[:5]:
    print(f"\n{'='*60}")
    print("Fayl:", file.name)
    print("Ölçü:", round(file.stat().st_size / 1024, 2), "KB")

    try:
        with open(file, "r", encoding="utf-8-sig") as f:
            data = json.load(f)

        print("Data type:", type(data).__name__)

        if isinstance(data, dict):
            print("Keys:", list(data.keys())[:30])
            print("Sample:", str(data)[:1500])

        elif isinstance(data, list):
            print("Element sayı:", len(data))

            if data:
                print("Birinci element:", str(data[0])[:1500])

        else:
            print("Məzmun:", str(data)[:1500])

    except Exception as e:
        print("Oxuma xətası:", e)


JSON fayllarının sayı: 0

ƏN BÖYÜK 5 JSON FAYLI:


In [2]:
from pathlib import Path

project = Path.cwd()

print("Cari qovluq:", project)
print("\nData qovluqları:")

for p in project.rglob("*"):
    if p.is_file():
        print(p.relative_to(project))

Cari qovluq: c:\Users\User\Desktop\EduInsight-AI\notebooks\notebooks

Data qovluqları:
05_text_preprocessing.ipynb
notebooks\06_tfidf_feature_extraction.ipynb
notebooks\notebooks\07_sentiment_classification.ipynb
notebooks\notebooks\08_topic_analysis.ipynb
notebooks\notebooks\09_absa_analysis.ipynb
notebooks\notebooks\10_model_evaluation.ipynb
notebooks\notebooks\11_dataset_annotation_inspection.ipynb
notebooks\notebooks\12_absa_model_training.ipynb


In [3]:

from pathlib import Path

current = Path.cwd()

project = next(
    (p for p in [current, *current.parents]
     if (p / "data" / "raw").exists()),
    None
)

if project is None:
    print("data/raw qovluğu tapılmadı!")
else:
    raw_folder = project / "data" / "raw"

    print("PROJECT:", project)
    print("DATASET PATH:", raw_folder)

    files = [f for f in raw_folder.rglob("*") if f.is_file()]

    print("Ümumi fayl sayı:", len(files))
    print("\nƏn böyük 10 fayl:")

    for f in sorted(files, key=lambda x: x.stat().st_size, reverse=True)[:10]:
        print(f"{f.name} | {f.suffix} | {f.stat().st_size / 1024:.2f} KB")


PROJECT: c:\Users\User\Desktop\EduInsight-AI
DATASET PATH: c:\Users\User\Desktop\EduInsight-AI\data\raw
Ümumi fayl sayı: 506

Ən böyük 10 fayl:
EduRABSA_Dataset.zip | .zip | 30440.59 KB
1_He_2020_RMP_v2_with_sampling_criteria.csv | .csv | 14128.34 KB
2_Uwaterloo_course_reviews_with_sampling_criteria.csv | .csv | 11134.71 KB
Train_val_Merged_AEOE_4000.txt | .txt | 7918.02 KB
DEV_TEST_Merged_AEOE_2500.txt | .txt | 5220.35 KB
Train_validation_Merged_ASQE_N4000.json | .json | 5052.69 KB
Train_val_Merged_OEC_4000.txt | .txt | 5024.64 KB
DEV_TEST_Merged_OEC_2500.txt | .txt | 3303.63 KB
DEV_TEST_Merged_ASQE_N2500.json | .json | 3228.20 KB
Train_val_Merged_APC_4000.txt | .txt | 3122.08 KB


In [4]:

import json
from pathlib import Path

project = Path.cwd()

while not (project / "data" / "raw").exists():
    project = project.parent

raw = project / "data" / "raw"

target_files = [
    "Train_validation_Merged_ASQE_N4000.json",
    "DEV_TEST_Merged_ASQE_N2500.json"
]

for filename in target_files:
    path = raw / filename

    print("\n" + "=" * 70)
    print("FILE:", filename)

    with open(path, "r", encoding="utf-8-sig") as f:
        data = json.load(f)

    print("TYPE:", type(data).__name__)

    if isinstance(data, list):
        print("TOTAL RECORDS:", len(data))

        for i, item in enumerate(data[:3]):
            print(f"\nRECORD {i + 1}:")
            print(json.dumps(item, ensure_ascii=False, indent=2)[:2500])

    elif isinstance(data, dict):
        print("KEYS:", list(data.keys()))

        for key, value in list(data.items())[:5]:
            print(f"\nKEY: {key}")
            print("VALUE:", str(value)[:1500])



FILE: Train_validation_Merged_ASQE_N4000.json


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\User\\Desktop\\EduInsight-AI\\data\\raw\\Train_validation_Merged_ASQE_N4000.json'

In [5]:
from pathlib import Path

project = Path.cwd()

while not (project / "data" / "raw").exists():
    project = project.parent

raw = project / "data" / "raw"

for f in raw.rglob("*ASQE*"):
    if f.is_file():
        print(f.name)
        print(f)

ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21-12)_(FULL_checked)_(V2).json
c:\Users\User\Desktop\EduInsight-AI\data\raw\ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21-12)_(FULL_checked)_(V2).json
ASQEJsonFile_(N=100_batch=10)_(2024-11-21_23-08-33)_(FULL_checked).json
c:\Users\User\Desktop\EduInsight-AI\data\raw\ASQEJsonFile_(N=100_batch=10)_(2024-11-21_23-08-33)_(FULL_checked).json
ASQEJsonFile_(N=100_batch=11)_(2024-11-22_11-27-55)_(FULL_checked).json
c:\Users\User\Desktop\EduInsight-AI\data\raw\ASQEJsonFile_(N=100_batch=11)_(2024-11-22_11-27-55)_(FULL_checked).json
ASQEJsonFile_(N=100_batch=12)_(2024-11-22_11-27-55)_(FULL_checked).json
c:\Users\User\Desktop\EduInsight-AI\data\raw\ASQEJsonFile_(N=100_batch=12)_(2024-11-22_11-27-55)_(FULL_checked).json
ASQEJsonFile_(N=100_batch=13)_(2024-11-22_12-24-24)_(FULL_checked).json
c:\Users\User\Desktop\EduInsight-AI\data\raw\ASQEJsonFile_(N=100_batch=13)_(2024-11-22_12-24-24)_(FULL_checked).json
ASQEJsonFile_(N=100_batch=14)_(2024-11-23_10-41

In [6]:

import json
from pathlib import Path

project = Path.cwd()

while not (project / "data" / "raw").exists():
    project = project.parent

raw = project / "data" / "raw"

files = [
    f for f in raw.rglob("*.json")
    if "ASQEJsonFile" in f.name
]

print("ASQE fayllarının sayı:", len(files))

if files:
    path = files[0]

    print("\nNümunə fayl:", path.name)

    with open(path, "r", encoding="utf-8-sig") as f:
        data = json.load(f)

    print("\nData type:", type(data).__name__)

    if isinstance(data, list):
        print("Element sayı:", len(data))
        if data:
            print("\nİlk element:")
            print(json.dumps(data[0], ensure_ascii=False, indent=2)[:3000])

    elif isinstance(data, dict):
        print("Keys:", list(data.keys())[:30])
        print("\nNümunə:")
        print(json.dumps(data, ensure_ascii=False, indent=2)[:3000])


ASQE fayllarının sayı: 65

Nümunə fayl: ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21-12)_(FULL_checked)_(V2).json

Data type: dict
Keys: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30']

Nümunə:
{
  "1": {
    "id": "1",
    "original_id": "1_65_2377_7868",
    "text": "She will only accept. docx forms in microsoft 2007. But, surprise, mason doesn't use vista or microsoft 2007! They still use XP, and microsoft 2006. This professor is useless and unreasonable. TAKE ANOTHER PROFESSOR",
    "Quadruplets": [
      {
        "aspect": "This professor",
        "opinion": "useless",
        "category": "Staff - Overall",
        "sentiment": "Negative"
      },
      {
        "aspect": "This professor",
        "opinion": "unreasonable",
        "category": "Staff - Attitude",
        "sentiment": "Negative"
      },
      {
        "aspect": "null",
        "opinion"

In [7]:

import json
import pandas as pd
from pathlib import Path

# Project root folder
project = Path.cwd()

while not (project / "data" / "raw").exists():
    if project.parent == project:
        raise FileNotFoundError("Project root folder not found.")
    project = project.parent

raw_folder = project / "data" / "raw"
output_folder = project / "results" / "absa"
output_folder.mkdir(parents=True, exist_ok=True)

# Find all ASQE JSON files
json_files = [
    file for file in raw_folder.rglob("*.json")
    if "ASQEJsonFile" in file.name
]

reviews = []
quadruplets = []

# Process every file
for file_path in json_files:

    with open(file_path, "r", encoding="utf-8-sig") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        continue

    for record_key, record in data.items():

        if not isinstance(record, dict):
            continue

        review_text = record.get("text")

        if not isinstance(review_text, str) or not review_text.strip():
            continue

        review_id = record.get("id", record_key)
        original_id = record.get("original_id")

        review_row = {
            "review_id": review_id,
            "original_id": original_id,
            "text": review_text.strip(),
            "source_file": file_path.name
        }

        reviews.append(review_row)

        annotations = record.get("Quadruplets", [])

        if not isinstance(annotations, list):
            continue

        for annotation in annotations:

            if not isinstance(annotation, dict):
                continue

            quadruplets.append({
                "review_id": review_id,
                "original_id": original_id,
                "text": review_text.strip(),
                "aspect": annotation.get("aspect"),
                "opinion": annotation.get("opinion"),
                "category": annotation.get("category"),
                "sentiment": annotation.get("sentiment"),
                "source_file": file_path.name
            })

# Create DataFrames
reviews_df = pd.DataFrame(reviews)
quadruplets_df = pd.DataFrame(quadruplets)

# Save CSV files
reviews_path = output_folder / "asqe_reviews.csv"
quadruplets_path = output_folder / "asqe_quadruplets.csv"

reviews_df.to_csv(reviews_path, index=False, encoding="utf-8-sig")
quadruplets_df.to_csv(quadruplets_path, index=False, encoding="utf-8-sig")

# Summary
print("=" * 50)
print("ASQE DATASET PROCESSING COMPLETED")
print("=" * 50)

print("JSON files processed:", len(json_files))
print("Total reviews:", len(reviews_df))
print("Total quadruplets:", len(quadruplets_df))

if not quadruplets_df.empty:
    print("\nSentiment distribution:")
    print(quadruplets_df["sentiment"].value_counts(dropna=False))

    print("\nTop 10 categories:")
    print(quadruplets_df["category"].value_counts().head(10))

print("\nSaved files:")
print(reviews_path)
print(quadruplets_path)

print("\nSample annotations:")
display(quadruplets_df.head(10))


ASQE DATASET PROCESSING COMPLETED
JSON files processed: 65
Total reviews: 6500
Total quadruplets: 27037

Sentiment distribution:
sentiment
Positive    15575
Negative     8700
Neutral      2762
Name: count, dtype: int64

Top 10 categories:
category
Staff - Teaching              4825
Course - Overall              3933
Staff - Overall               3144
Course - Assessment           2528
Course - Content              2274
Course - Difficulty           2228
Staff - Attitude              1566
Course - Learning activity    1368
Staff - Personal traits        923
Staff - Helpfulness            820
Name: count, dtype: int64

Saved files:
c:\Users\User\Desktop\EduInsight-AI\results\absa\asqe_reviews.csv
c:\Users\User\Desktop\EduInsight-AI\results\absa\asqe_quadruplets.csv

Sample annotations:


,review_id,original_id,text,aspect,opinion,category,sentiment,source_file
0,1,1_65_2377_7868,She will only accept. docx forms in microsoft ...,This professor,useless,Staff - Overall,Negative,ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21...
1,1,1_65_2377_7868,She will only accept. docx forms in microsoft ...,This professor,unreasonable,Staff - Attitude,Negative,ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21...
2,1,1_65_2377_7868,She will only accept. docx forms in microsoft ...,null,TAKE ANOTHER PROFESSOR,Staff - Overall,Negative,ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21...
3,2,1_82_24_8586,Great Teacher! Learned a lot! If coming to cla...,Teacher,Great,Staff - Overall,Positive,ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21...
4,2,1_82_24_8586,Great Teacher! Learned a lot! If coming to cla...,Teacher,Good stuff,Staff - Overall,Positive,ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21...
5,2,1_82_24_8586,Great Teacher! Learned a lot! If coming to cla...,null,Learned a lot,Staff - Teaching,Positive,ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21...
6,2,1_82_24_8586,Great Teacher! Learned a lot! If coming to cla...,Teacher,take her,Staff - Overall,Positive,ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21...
7,2,1_82_24_8586,Great Teacher! Learned a lot! If coming to cla...,Teacher,know her stuff,Staff - Knowledge & skills,Positive,ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21...
8,2,1_82_24_8586,Great Teacher! Learned a lot! If coming to cla...,Teacher,Very accommodating and understanding,Staff - Attitude,Positive,ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21...
9,2,1_82_24_8586,Great Teacher! Learned a lot! If coming to cla...,Teacher,Highly recommended,Staff - Overall,Positive,ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21...
